<a href="https://colab.research.google.com/github/M1tayka/PIRSMA_M/blob/dz1/dz1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:


!wget -q https://archive.apache.org/dist/spark/spark-3.4.1/spark-3.4.1-bin-hadoop3.tgz
!tar xf spark-3.4.1-bin-hadoop3.tgz
!pip install -q findspark

import findspark
findspark.init("/content/spark-3.4.1-bin-hadoop3")

import csv
import math
from pyspark import SparkContext

sc = SparkContext.getOrCreate()



In [5]:
from google.colab import files
uploaded = files.upload()


Saving links.csv to links.csv
Saving movies.csv to movies.csv
Saving ratings.csv to ratings.csv
Saving tags.csv to tags.csv


In [6]:
def parse_csv_line(line):

    reader = csv.reader([line])
    return next(reader)

def cosine_similarity(target_vec_dict, target_norm, compare_vec_list):

    compare_vec_dict = dict(compare_vec_list)

    common_users = set(target_vec_dict.keys()) & set(compare_vec_dict.keys())

    if not common_users:
        return 0.0

    dot_product = sum(target_vec_dict[u] * compare_vec_dict[u] for u in common_users)

    compare_norm = math.sqrt(sum(r*r for r in compare_vec_dict.values()))

    if target_norm == 0 or compare_norm == 0:
        return 0.0

    return dot_product / (target_norm * compare_norm)

In [7]:
movies_rdd = sc.textFile("movies.csv") \
    .filter(lambda x: not x.startswith("movieId")) \
    .map(parse_csv_line) \
    .map(lambda x: (int(x[0]), x[1]))

ratings_rdd = sc.textFile("ratings.csv") \
    .filter(lambda x: not x.startswith("userId")) \
    .map(parse_csv_line) \
    .map(lambda x: (int(x[1]), (int(x[0]), float(x[2]))))

movie_vectors_rdd = ratings_rdd.groupByKey().mapValues(list).cache()

In [8]:
TARGET_MOVIE_ID = 589

target_data = movie_vectors_rdd.filter(lambda x: x[0] == TARGET_MOVIE_ID).collect()

if not target_data:
    print(f"ошибка - фильм с id {TARGET_MOVIE_ID} не найден в рейтингах")
else:
    target_list = target_data[0][1]
    target_dict = dict(target_list)
    target_norm = math.sqrt(sum(r*r for r in target_dict.values()))

    similarity_rdd = movie_vectors_rdd \
        .filter(lambda x: x[0] != TARGET_MOVIE_ID) \
        .map(lambda x: (x[0], cosine_similarity(target_dict, target_norm, x[1]))) \
        .filter(lambda x: x[1] > 0)

    top_10_similar = similarity_rdd.takeOrdered(10, key=lambda x: -x[1])

    titles_map = movies_rdd.collectAsMap()


In [26]:
target_title = titles_map.get(TARGET_MOVIE_ID, "Unknown Title")
print("_" * 80)
print(f"\nфильм - {target_title} (id - {TARGET_MOVIE_ID})")
print(f"\nтоп 10 наиболее похожих фильмов:")
print("_" * 80)
print(f"{'id':<6} | {'cходство':<10} | {'название'}")
print("_" * 80)

for movie_id, score in top_10_similar:
        title = titles_map.get(movie_id, "Title not found")
        print(f"{movie_id:<6} | {score:<10.6f} | {title}")

________________________________________________________________________________

фильм - Terminator 2: Judgment Day (1991) (id - 589)

топ 10 наиболее похожих фильмов:
________________________________________________________________________________
id     | cходство   | название
________________________________________________________________________________
480    | 0.719983   | Jurassic Park (1993)
1240   | 0.695724   | Terminator, The (1984)
110    | 0.659827   | Braveheart (1995)
592    | 0.645603   | Batman (1989)
457    | 0.637561   | Fugitive, The (1993)
377    | 0.630092   | Speed (1994)
1196   | 0.618530   | Star Wars: Episode V - The Empire Strikes Back (1980)
380    | 0.611164   | True Lies (1994)
296    | 0.610284   | Pulp Fiction (1994)
356    | 0.600886   | Forrest Gump (1994)
